# nb_04_batch_score - score the open orders with PREDICT

Runs every morning as the last step of `pl_nightly_delivery_promise`. It reads `gold_open_orders_features` (written by `nb_02_features`
in `open_orders` mode), scores it with the registered model through `MLFlowTransformer` (Fabric's PREDICT), and writes
`gold_open_orders_scored` with `late_risk`, a `risk_band` and `scored_at`.

Compare it with the notebook the **Apply this version › Apply this model in wizard** button generates: same transformer, we just add the band and the timestamp.

**Parameters.** `model_version` 0 means "the newest registered version" (after Lab 7 that is version 2).

In [ ]:
model_name = "mdl_delivery_promise"
model_version = 0            # 0 = latest registered version; or pin a number, e.g. 1
input_table = "gold_open_orders_features"
output_table = "gold_open_orders_scored"

**Imports and the contract columns**, in order.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType
from synapse.ml.predict import MLFlowTransformer

spark.conf.set("spark.synapse.ml.predict.enabled", "true")

# The 16 model inputs, in contract order (kit/contracts/feature_contract.json). Order matters: requests use orientation "values".
FEATURES = ["carrier_id", "ship_method_express", "warehouse_id", "region_id", "remoteness", "distance_km", "order_hour",
            "order_dow", "is_weekend", "is_peak", "items_count", "total_weight_kg", "order_value", "promised_days",
            "backlog_index", "forecast_severity"]
FLOAT_COLS = ["distance_km", "total_weight_kg", "order_value", "backlog_index"]   # double
INT_COLS = [c for c in FEATURES if c not in FLOAT_COLS]                           # long

**Resolve the version.** If `model_version` is 0, ask the registry for the highest version number.

In [ ]:
if int(model_version) <= 0:
    versions = MlflowClient().search_model_versions(f"name='{model_name}'")
    model_version = max(int(v.version) for v in versions)
model_version = int(model_version)
print(f"scoring with {model_name} version {model_version}")

**Load the features** and cast to exactly the signature types (long / double), in contract order.

In [ ]:
src = spark.read.table(input_table)
casts = [F.col(c).cast("double" if c in FLOAT_COLS else "long").alias(c) for c in FEATURES]
feats = src.select("order_id", "order_value", "ship_method", "promised_date", *casts)
print(feats.count(), "open orders to score")

**PREDICT.** The transformer loads the registered MLflow model and adds one output column.

In [ ]:
model = MLFlowTransformer(inputCols=FEATURES, outputCol="late_risk", modelName=model_name, modelVersion=model_version)
pred = model.transform(feats)
pred.printSchema()
pred.select("order_id", "late_risk").show(5, truncate=False)

**⟲ verify at recording (kit open check 14):** `late_risk` must be ONE probability per row (0-1), not a 0/1 label. A native Booster gives probabilities; a scikit-learn-API classifier would give labels. If PREDICT returns an array column, the cell takes its first element.

In [ ]:
if isinstance(pred.schema["late_risk"].dataType, ArrayType):
    pred = pred.withColumn("late_risk", F.col("late_risk")[0])
pred = pred.withColumn("late_risk", F.col("late_risk").cast("double"))
stats = pred.agg(F.min("late_risk"), F.max("late_risk"), F.countDistinct("late_risk")).collect()[0]
print(f"late_risk min {stats[0]:.3f}  max {stats[1]:.3f}  distinct values {stats[2]}")
assert stats[2] > 2, "late_risk looks like a 0/1 label: register the native Booster (lgb.train), not a classifier"

**Risk band and timestamp**, then write `gold_open_orders_scored`. Bands follow the app's decision policy: low < 0.35, medium 0.35-0.6, high ≥ 0.6.

In [ ]:
scored = (pred
    .withColumn("risk_band", F.when(F.col("late_risk") >= 0.6, "high")
                              .when(F.col("late_risk") >= 0.35, "medium").otherwise("low"))
    .withColumn("scored_at", F.current_timestamp())
    .withColumn("model_version", F.lit(model_version))
    .select("order_id", "order_value", "ship_method", "promised_date", *FEATURES, "late_risk", "risk_band", "scored_at", "model_version"))
scored.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
print(f"wrote {output_table}: {scored.count()} rows")

**Check cell.** How many orders fall in each band, and the at-risk revenue the Power BI card will show.

In [ ]:
(spark.read.table(output_table).groupBy("risk_band")
  .agg(F.count("*").alias("orders"), F.round(F.sum("order_value"), 2).alias("revenue"), F.round(F.avg("late_risk"), 3).alias("avg_risk"))
  .orderBy("risk_band").show())